# M04 — 持久化與記憶

本 notebook 對應 `README.md`，逐格執行即可。

我們會在 M02 學過的 `MessagesState` 聊天圖上掛一個 **checkpointer**，
讓圖能跨多次 `invoke` 記得對話；再用 `thread_id` 示範「記得」與「忘記」，
最後換成 `SqliteSaver` 做本機持久化，並用 `get_state` 觀察 checkpoint。

## 1. 環境準備

載入共用 helper，取得供應商無關的 chat model。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 先建一張「沒有記憶」的聊天圖

這就是 M02 的 `MessagesState` 圖：一個 node 把整串對話餵給模型，
回覆透過 `add_messages` reducer 累加進 `messages`。
注意這一格**先不掛 checkpointer**，等下用來對照。

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState

def chatbot(state: MessagesState) -> dict:
    # Feed the whole conversation so far to the model; return its reply.
    # add_messages reducer will append (not overwrite) this into `messages`.
    reply = model.invoke(state["messages"])
    return {"messages": [reply]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph_no_memory = builder.compile()   # no checkpointer -> no memory

## 3. 證明它沒有記憶

兩次獨立 `invoke`，第二次問名字。因為沒有 checkpointer，第二次是全新開始，
圖看不到第一次說過的話。

Expected output: 第二輪模型答不出名字（會說不知道之類）。

In [ ]:
graph_no_memory.invoke({"messages": [{"role": "user", "content": "嗨，我叫小明"}]})

second = graph_no_memory.invoke(
    {"messages": [{"role": "user", "content": "我剛剛說我叫什麼名字？"}]}
)
print(second["messages"][-1].content)
# Expected: model has no idea -- the first invoke's state was thrown away.

## 4. 掛上 InMemorySaver：給圖裝記憶

同一個 `builder`，這次 compile 時掛一個 `InMemorySaver`。
它把每一步的 state 存成 checkpoint，按 `thread_id` 分組。
教學用：存在 RAM，程式結束就忘。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

graph = builder.compile(checkpointer=InMemorySaver())

## 5. 同一個 thread_id：第二輪記得名字

關鍵在 config 裡的 `thread_id`。兩次 `invoke` 都用 `"alice"` 這條對話線，
第二輪會先載回第一輪的 checkpoint，於是模型看得到「我叫小明」。

Expected output: 模型答得出「小明」。

In [ ]:
config_alice = {"configurable": {"thread_id": "alice"}}

graph.invoke(
    {"messages": [{"role": "user", "content": "嗨，我叫小明"}]},
    config_alice,
)

answer = graph.invoke(
    {"messages": [{"role": "user", "content": "我剛剛說我叫什麼名字？"}]},
    config_alice,
)
print(answer["messages"][-1].content)
# Expected: contains "小明" -- the checkpointer fed the first turn back in.

## 6. 換一個 thread_id：它忘記了

同一張圖、同一個 checkpointer，但換成 `thread_id="bob"`。
這是一條全新的對話線，看不到 alice 的歷史 → 答不出名字。

Expected output: bob 這條線不知道「小明」。

In [ ]:
config_bob = {"configurable": {"thread_id": "bob"}}

reply_bob = graph.invoke(
    {"messages": [{"role": "user", "content": "我剛剛說我叫什麼名字？"}]},
    config_bob,
)
print(reply_bob["messages"][-1].content)
# Expected: model doesn't know -- different thread = isolated timeline.

## 🧪 練習 1

在 `config_alice` 這條對話線上，再接一輪問模型：
「請用一句話總結我們目前聊了什麼？」
觀察它是否引用了前面提過的名字。
接著把 `thread_id` 改成你自己的名字，開一條全新對話再問同樣的問題，
對照兩條線的差異。

In [ ]:
# Your code here.
# Hint:
# graph.invoke(
#     {"messages": [{"role": "user", "content": "請用一句話總結我們目前聊了什麼？"}]},
#     config_alice,
# )

## 7. 用 get_state 觀察當前 checkpoint

掛了 checkpointer 後，可以隨時把某個 thread 的狀態挖出來看。
`get_state` 回傳一個 `StateSnapshot`：`.values` 是當下 state、
`.next` 是接下來要跑的 node（跑完一輪通常是空的）。

Expected output: alice 這條線累積的多則訊息。

In [ ]:
snapshot = graph.get_state(config_alice)
print("訊息數量:", len(snapshot.values["messages"]))
print("下一步要跑的 node:", snapshot.next)   # Expected: () -- nothing pending
for m in snapshot.values["messages"]:
    print(f"  [{m.type}] {m.content}")

## 8. 用 get_state_history 看歷史 checkpoint

`get_state_history` 從新到舊列出每一個 checkpoint。
每個 snapshot 帶著自己的 checkpoint id（藏在 `.config` 裡）。
M05 的「時光旅行」就是拿這些 id，讓圖從過去某一步重新跑。

Expected output: 多個 checkpoint，每次 invoke 都會新增。

In [ ]:
history = list(graph.get_state_history(config_alice))
print("checkpoint 數量:", len(history))
for snap in history:
    checkpoint_id = snap.config["configurable"]["checkpoint_id"]
    print(f"  checkpoint {checkpoint_id[:8]}... 訊息數={len(snap.values['messages'])}")

## 9. 本機持久化：SqliteSaver

`InMemorySaver` 程式一結束記憶就蒸發。要重啟後仍記得，換成 `SqliteSaver`：
它把 checkpoint 寫進本機 `.sqlite` 檔。API 完全一樣，圖的程式碼一行都不改。

`SqliteSaver` 是 context manager，要在 `with` 區塊內使用。
第一次跑會建立 `memory.sqlite`；下次重新啟動 Python、用同一個 `thread_id`，
仍然讀得回這段對話。

Expected output: 模型答得出「阿華」，且記憶寫進了本機檔案。

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

config_db = {"configurable": {"thread_id": "persistent-user"}}

with SqliteSaver.from_conn_string("memory.sqlite") as checkpointer:
    db_graph = builder.compile(checkpointer=checkpointer)

    db_graph.invoke(
        {"messages": [{"role": "user", "content": "你好，我叫阿華"}]},
        config_db,
    )
    out = db_graph.invoke(
        {"messages": [{"role": "user", "content": "我叫什麼名字？"}]},
        config_db,
    )
    print(out["messages"][-1].content)
    # Expected: contains "阿華".
    # Re-run the whole program later with the same thread_id and it still remembers,
    # because the checkpoints live in memory.sqlite on disk.

## 🧪 練習 2

重啟 kernel（讓 RAM 清空），**只跑第 1 格與這一格**，
用同一個 `thread_id="persistent-user"` 連到 `memory.sqlite`，
不要重新自我介紹，直接問「我叫什麼名字？」。
驗證 `SqliteSaver` 真的把記憶寫進了硬碟、重啟後仍記得。

In [ ]:
# Your code here.
# Hint:
# from langgraph.checkpoint.sqlite import SqliteSaver
# config_db = {"configurable": {"thread_id": "persistent-user"}}
# with SqliteSaver.from_conn_string("memory.sqlite") as cp:
#     g = builder.compile(checkpointer=cp)
#     out = g.invoke({"messages": [{"role": "user", "content": "我叫什麼名字？"}]}, config_db)
#     print(out["messages"][-1].content)

## 小結 & 下一步

- **checkpointer** 把每一步 state 存成 checkpoint，圖因此有了短期記憶。
- **`thread_id`** 把 checkpoint 按對話分組：同 thread 記得、換 thread 忘記。
- **`InMemorySaver`** 拿來學（存 RAM）、**`SqliteSaver`** 拿來持久化（存本機檔，重啟仍記得）。
- **`get_state` / `get_state_history`** 讓你觀察當前與歷史 checkpoint。

短期記憶（checkpointer + thread）解決「同一場對話的延續」；長期記憶（store）則
負責「跨對話要永久記住的知識」，是另一個獨立機制，這裡先建立概念即可。

有了 checkpoint，圖不只能「記得」，還能「回到過去某一步重新跑」——這正是下一個模組
**M05 Human-in-the-loop** 的基礎：用 `interrupt` 在關鍵步驟暫停、等人類審核或修改，
再從那個 checkpoint 續跑。下一站見。